# Future Returns Calculation for Cryptocurrency Symbols

This notebook calculates future returns for various time horizons using hourly OHLCV data.

**Input**: `/home/craz/crypto/crypto-data/pickle_hour_cache/usdt_data_YYYY-MM.pkl` (hourly data)

**Output**: `/home/craz/crypto/crypto-data/future_returns/future_returns_YYYY-MM.pkl` (monthly returns)

**Return Calculation**: `(future_open_price / current_open_price) - 1`

**Time Horizons**: 1h,2h,3h,4h,5h,6h,7h,8h,10h,12h,14h,16h,18h,20h,24h,28h,32h,36h,42h,48h,54h,60h,72h,84h,96h,108h,120h,144h

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import glob
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [2]:
# Configuration
INPUT_PATH = '/home/craz/crypto/crypto-data/filter_hour_cache/'
OUTPUT_PATH = '/home/craz/crypto/crypto-data/future_returns/'

# Time horizons in hours
TIME_HORIZONS = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 16, 18, 20, 24, 
                 28, 32, 36, 42, 48, 54, 60, 72, 84, 96, 108, 120, 144]

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Output directory created: {os.path.exists(OUTPUT_PATH)}")
print(f"Time horizons ({len(TIME_HORIZONS)}): {TIME_HORIZONS}")

Input path: /home/craz/crypto/crypto-data/filter_hour_cache/
Output path: /home/craz/crypto/crypto-data/future_returns/
Output directory created: True
Time horizons (28): [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 16, 18, 20, 24, 28, 32, 36, 42, 48, 54, 60, 72, 84, 96, 108, 120, 144]


In [3]:
def get_available_months():
    """
    Get all available monthly pickle files and extract year-month strings.
    
    Returns:
        list: Sorted list of year-month strings (e.g., ['2020-01', '2020-02', ...])
    """
    files = glob.glob(os.path.join(INPUT_PATH, "usdt_data_*.pkl"))
    months = []
    
    for file in files:
        filename = os.path.basename(file)
        # Extract YYYY-MM from usdt_data_YYYY-MM.pkl
        year_month = filename.replace('usdt_data_', '').replace('.pkl', '')
        months.append(year_month)
    
    months.sort()
    return months

available_months = get_available_months()
print(f"Found {len(available_months)} available months")
if available_months:
    print(f"First: {available_months[0]}")
    print(f"Last: {available_months[-1]}")
    print(f"Recent months: {available_months[-5:]}")

Found 70 available months
First: 2020-01
Last: 2025-10
Recent months: ['2025-06', '2025-07', '2025-08', '2025-09', '2025-10']


In [4]:
def load_monthly_data(year_month):
    """
    Load hourly data for a specific month with pickle protocol compatibility.
    
    Args:
        year_month (str): Year-month string in format 'YYYY-MM'
    
    Returns:
        DataFrame: Hourly OHLCV data with MultiIndex (open_time, symbol)
    """
    file_path = os.path.join(INPUT_PATH, f"usdt_data_{year_month}.pkl")
    
    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}")
        return None
    
    try:
        # First try pandas read_pickle
        df = pd.read_pickle(file_path)
        print(f"✅ Loaded {year_month}: {len(df)} rows, {len(df.index.get_level_values('symbol').unique())} symbols")
        return df
    except (ValueError, ImportError) as e:
        if "pickle protocol" in str(e):
            print(f"⚠️ Pickle protocol issue for {year_month}, trying alternative methods...")
            
            # Method 1: Try with standard pickle
            try:
                with open(file_path, 'rb') as f:
                    df = pickle.load(f)
                print(f"✅ Loaded {year_month} with pickle.load: {len(df)} rows, {len(df.index.get_level_values('symbol').unique())} symbols")
                return df
            except Exception as e2:
                print(f"❌ pickle.load failed: {e2}")
            
            # Method 2: Try with joblib if available
            try:
                import joblib
                df = joblib.load(file_path)
                print(f"✅ Loaded {year_month} with joblib: {len(df)} rows, {len(df.index.get_level_values('symbol').unique())} symbols")
                return df
            except ImportError:
                print(f"❌ joblib not available")
            except Exception as e3:
                print(f"❌ joblib.load failed: {e3}")
            
            print(f"❌ All loading methods failed for {year_month}. File may be incompatible with Python 3.7")
            print(f"💡 Suggestion: Try running this notebook with Python 3.8+ for better pickle compatibility")
            return None
        else:
            print(f"❌ Error loading {year_month}: {e}")
            return None
    except Exception as e:
        print(f"❌ Unexpected error loading {year_month}: {e}")
        return None

# Test loading a sample month
if available_months:
    test_month = available_months[0]  # Try with first (oldest) month
    print(f"\nTesting with month: {test_month}")
    test_df = load_monthly_data(test_month)
    
    if test_df is not None:
        print(f"Data shape: {test_df.shape}")
        print(f"Columns: {test_df.columns.tolist()}")
        print(f"Time range: {test_df.index.get_level_values('open_time').min()} to {test_df.index.get_level_values('open_time').max()}")
        print(f"\nFirst few rows:")
        print(test_df.head())
    else:
        print(f"❌ Could not load test data. The pickle files may be incompatible with Python 3.7")
        print(f"💡 To resolve this issue:")
        print(f"   1. Use Python 3.8+ to run this notebook")
        print(f"   2. Or regenerate the pickle files with Python 3.7 compatible protocol")
        print(f"   3. Alternative: Create sample data for testing")


Testing with month: 2020-01
✅ Loaded 2020-01: 4760 rows, 10 symbols
Data shape: (4760, 5)
Columns: ['open', 'high', 'low', 'close', 'volume']
Time range: 2020-01-01 00:00:00 to 2020-01-31 23:00:00

First few rows:
                                open     high      low    close     volume
open_time           symbol                                                
2020-01-01 00:00:00 BCHUSDT   206.58   207.92   206.15   206.82  13171.038
                    BTCUSDT  7191.80  7221.75  7184.46  7207.71   5079.656
                    ETHUSDT   129.61   130.51   129.06   130.01  41335.565
2020-01-01 01:00:00 BCHUSDT   206.82   207.28   204.15   204.28  19502.176
                    BTCUSDT  7207.78  7208.14  7146.73  7163.61   7237.620


In [5]:
def get_next_month(year_month):
    """
    Get the next month string given a year-month string.
    
    Args:
        year_month (str): Year-month string in format 'YYYY-MM'
    
    Returns:
        str: Next month string in format 'YYYY-MM'
    """
    year, month = map(int, year_month.split('-'))
    
    if month == 12:
        next_year = year + 1
        next_month = 1
    else:
        next_year = year
        next_month = month + 1
    
    return f"{next_year:04d}-{next_month:02d}"

# Test the function
test_cases = ['2024-11', '2024-12', '2025-01', '2025-06']
for test_case in test_cases:
    next_month = get_next_month(test_case)
    print(f"{test_case} → {next_month}")

2024-11 → 2024-12
2024-12 → 2025-01
2025-01 → 2025-02
2025-06 → 2025-07


In [6]:
def calculate_future_returns(df, time_horizons):
    """
    Calculate future returns for all symbols at various time horizons.
    
    Args:
        df (DataFrame): Hourly OHLCV data with MultiIndex (open_time, symbol)
        time_horizons (list): List of time horizons in hours
    
    Returns:
        DataFrame: Returns data with MultiIndex (open_time, symbol) and return columns
    """
    print(f"Calculating future returns for {len(time_horizons)} time horizons...")
    
    # Reset index to work with datetime operations
    df_reset = df.reset_index()
    
    # Ensure open_time is datetime
    df_reset['open_time'] = pd.to_datetime(df_reset['open_time'])
    
    # Create a list to store results for each symbol
    results = []
    
    # Get unique symbols
    symbols = df_reset['symbol'].unique()
    print(f"Processing {len(symbols)} symbols...")
    
    # Process symbols with manual progress tracking
    for i, symbol in enumerate(symbols, 1):
        print(f"  [{i:2d}/{len(symbols)}] Processing {symbol}...", end="")
        
        # Get data for this symbol
        symbol_data = df_reset[df_reset['symbol'] == symbol].copy()
        symbol_data = symbol_data.sort_values('open_time')
        
        # Create result dataframe for this symbol
        symbol_results = symbol_data[['open_time', 'symbol', 'open']].copy()
        
        # Calculate returns for each time horizon
        for horizon in time_horizons:
            # Create future timestamp
            future_times = symbol_data['open_time'] + pd.Timedelta(hours=horizon)
            
            # Find future prices by merging with shifted data
            future_prices = []
            
            for j, (current_time, future_time) in enumerate(zip(symbol_data['open_time'], future_times)):
                # Find the exact future timestamp
                future_row = symbol_data[symbol_data['open_time'] == future_time]
                
                if len(future_row) > 0:
                    future_price = future_row['open'].iloc[0]
                    current_price = symbol_data['open'].iloc[j]
                    
                    # Calculate return: (future_price / current_price) - 1
                    if current_price > 0:
                        return_value = (future_price / current_price) - 1
                    else:
                        return_value = np.nan
                else:
                    # Future data not available
                    return_value = np.nan
                
                future_prices.append(return_value)
            
            # Add return column
            symbol_results[f'return_{horizon}h'] = future_prices
        
        results.append(symbol_results)
        print(" ✅")
    
    print(f"\nCombining results for all symbols...")
    
    # Combine all symbols
    final_df = pd.concat(results, ignore_index=True)
    
    # Set MultiIndex
    final_df = final_df.set_index(['open_time', 'symbol'])
    
    # Drop the 'open' column as we only need returns
    final_df = final_df.drop(columns=['open'])
    
    print(f"✅ Calculated returns for {len(final_df)} rows")
    print(f"Return columns: {[col for col in final_df.columns if col.startswith('return_')]}")
    
    return final_df

In [7]:
def process_monthly_returns(year_month, time_horizons=TIME_HORIZONS):
    """
    Process future returns for a single month.
    
    Args:
        year_month (str): Year-month string in format 'YYYY-MM'
        time_horizons (list): List of time horizons in hours
    
    Returns:
        bool: True if successful, False otherwise
    """
    print(f"\n{'='*60}")
    print(f"Processing returns for {year_month}")
    print(f"{'='*60}")
    
    # Load current month data
    current_df = load_monthly_data(year_month)
    if current_df is None:
        print(f"❌ Failed to load current month: {year_month}")
        return False
    
    # Load next month data for future returns calculation
    next_month = get_next_month(year_month)
    next_df = load_monthly_data(next_month)
    
    if next_df is None:
        print(f"⚠️ Next month data not available: {next_month}")
        print(f"Will only calculate returns within {year_month} (limited future data)")
        combined_df = current_df
    else:
        # Combine current and next month for complete future returns
        print(f"✅ Combining {year_month} and {next_month} data")
        combined_df = pd.concat([current_df, next_df])
        combined_df = combined_df.sort_index()
        print(f"Combined data shape: {combined_df.shape}")
    
    # Calculate future returns
    returns_df = calculate_future_returns(combined_df, time_horizons)
    
    # Filter to only keep current month's timestamps for output
    current_month_start = pd.to_datetime(f"{year_month}-01")
    if year_month.endswith('12'):
        next_year = int(year_month[:4]) + 1
        next_month_start = pd.to_datetime(f"{next_year}-01-01")
    else:
        month_num = int(year_month[-2:]) + 1
        next_month_start = pd.to_datetime(f"{year_month[:5]}{month_num:02d}-01")
    
    # Filter returns to current month only
    mask = (returns_df.index.get_level_values('open_time') >= current_month_start) & \
           (returns_df.index.get_level_values('open_time') < next_month_start)
    
    current_month_returns = returns_df[mask]
    
    print(f"Filtered to current month: {len(current_month_returns)} rows")
    
    # Save to pickle file
    output_file = os.path.join(OUTPUT_PATH, f"future_returns_{year_month}.pkl")
    
    try:
        current_month_returns.to_pickle(output_file)
        file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
        print(f"✅ Saved: {output_file}")
        print(f"File size: {file_size_mb:.2f} MB")
        
        # Basic validation
        print(f"\nValidation:")
        print(f"- Shape: {current_month_returns.shape}")
        print(f"- Return columns: {len([col for col in current_month_returns.columns if col.startswith('return_')])}")
        print(f"- Unique symbols: {len(current_month_returns.index.get_level_values('symbol').unique())}")
        
        # Check for any non-null returns
        return_cols = [col for col in current_month_returns.columns if col.startswith('return_')]
        non_null_counts = current_month_returns[return_cols].count()
        print(f"- Non-null return counts (first 5): {non_null_counts.head().to_dict()}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error saving: {e}")
        return False

## Create Sample Data for Testing (if needed)

Let's create sample data to test our implementation if the pickle files are incompatible.

## Test Future Returns Calculation with Sample Data

Let's test our returns calculation function with sample or real data.

## Batch Processing Function

Function to process multiple months in batch mode.

In [8]:
def batch_process_returns(start_month, end_month, time_horizons=TIME_HORIZONS):
    """
    Batch process future returns for a range of months.
    
    Args:
        start_month (str): Start month in format 'YYYY-MM'
        end_month (str): End month in format 'YYYY-MM' (inclusive)
        time_horizons (list): List of time horizons in hours
    
    Returns:
        dict: Processing results summary
    """
    print(f"\n{'='*80}")
    print(f"BATCH PROCESSING: {start_month} to {end_month}")
    print(f"{'='*80}")
    
    # Generate list of months to process
    start_year, start_mon = map(int, start_month.split('-'))
    end_year, end_mon = map(int, end_month.split('-'))
    
    months_to_process = []
    current_year, current_mon = start_year, start_mon
    
    while (current_year < end_year) or (current_year == end_year and current_mon <= end_mon):
        month_str = f"{current_year:04d}-{current_mon:02d}"
        months_to_process.append(month_str)
        
        # Move to next month
        if current_mon == 12:
            current_year += 1
            current_mon = 1
        else:
            current_mon += 1
    
    print(f"Months to process: {len(months_to_process)}")
    print(f"Month range: {months_to_process[0]} to {months_to_process[-1]}")
    
    # Filter to only available months
    available_to_process = [m for m in months_to_process if m in available_months]
    unavailable = [m for m in months_to_process if m not in available_months]
    
    print(f"Available to process: {len(available_to_process)}")
    if unavailable:
        print(f"⚠️ Unavailable months: {unavailable}")
    
    # Process each month
    results = {
        'successful': [],
        'failed': [],
        'skipped': unavailable
    }
    
    # Process months with manual progress tracking
    for i, month in enumerate(available_to_process, 1):
        print(f"\n[{i:2d}/{len(available_to_process)}] Processing {month}...")
        try:
            success = process_monthly_returns(month, time_horizons)
            
            if success:
                results['successful'].append(month)
                print(f"✅ {month} completed successfully")
            else:
                results['failed'].append(month)
                print(f"❌ {month} failed")
                
        except Exception as e:
            print(f"❌ Exception processing {month}: {e}")
            results['failed'].append(month)
    
    # Summary
    print(f"\n{'='*80}")
    print(f"BATCH PROCESSING COMPLETE")
    print(f"{'='*80}")
    print(f"Successful: {len(results['successful'])} months")
    print(f"Failed: {len(results['failed'])} months")
    print(f"Skipped: {len(results['skipped'])} months")
    
    if results['successful']:
        print(f"\n✅ Successfully processed:")
        for month in results['successful']:
            output_file = os.path.join(OUTPUT_PATH, f"future_returns_{month}.pkl")
            if os.path.exists(output_file):
                file_size = os.path.getsize(output_file) / (1024 * 1024)
                print(f"  {month}: {file_size:.2f} MB")
    
    if results['failed']:
        print(f"\n❌ Failed to process: {results['failed']}")
    
    if results['skipped']:
        print(f"\n⚠️ Skipped (unavailable): {results['skipped']}")
    
    return results

## Example: Batch Processing Recent Months

Process a small range of recent months as an example.

In [9]:
example_results = batch_process_returns('2025-04', '2025-08')


BATCH PROCESSING: 2025-04 to 2025-08
Months to process: 5
Month range: 2025-04 to 2025-08
Available to process: 5

[ 1/5] Processing 2025-04...

Processing returns for 2025-04
✅ Loaded 2025-04: 277044 rows, 398 symbols
✅ Loaded 2025-05: 305800 rows, 422 symbols
✅ Combining 2025-04 and 2025-05 data
Combined data shape: (582844, 5)
Calculating future returns for 28 time horizons...
Processing 422 symbols...
  [ 1/422] Processing 1000000MOGUSDT... ✅
  [ 2/422] Processing 1000BONKUSDT... ✅
  [ 3/422] Processing 1000CATUSDT... ✅
  [ 4/422] Processing 1000CHEEMSUSDT... ✅
  [ 5/422] Processing 1000FLOKIUSDT... ✅
  [ 6/422] Processing 1000LUNCUSDT... ✅
  [ 7/422] Processing 1000PEPEUSDT... ✅
  [ 8/422] Processing 1000RATSUSDT... ✅
  [ 9/422] Processing 1000SATSUSDT... ✅
  [10/422] Processing 1000SHIBUSDT... ✅
  [11/422] Processing 1000WHYUSDT... ✅
  [12/422] Processing 1000XECUSDT... ✅
  [13/422] Processing 1000XUSDT...

KeyboardInterrupt: 

## Full Batch Processing

**WARNING**: This will process ALL available months. Only run if you want to generate the complete dataset.

In [12]:
# UNCOMMENT THE LINES BELOW TO RUN FULL BATCH PROCESSING
# WARNING: This will take a long time and generate many large files

# if available_months:
#     # Process all available months (excluding the last one which might not have next month data)
#     full_start = available_months[0]
#     full_end = available_months[-2] if len(available_months) > 1 else available_months[-1]
#     
#     print(f"Full batch processing: {full_start} to {full_end}")
#     print(f"This will process {len(available_months)-1} months")
#     
#     # Confirm before processing
#     confirm = input("Are you sure you want to process all months? (yes/no): ")
#     
#     if confirm.lower() == 'yes':
#         full_results = batch_process_returns(full_start, full_end)
#         
#         print(f"\n🎉 Full Processing Complete!")
#         print(f"Successful: {len(full_results['successful'])}")
#         print(f"Failed: {len(full_results['failed'])}")
#         print(f"Skipped: {len(full_results['skipped'])}")
#     else:
#         print("Full processing cancelled.")
# else:
#     print("No months available for processing")

# print("Full batch processing code is commented out.")
# print("Uncomment the above code to run full processing.")

## List Generated Files

Show all generated future returns files.

In [13]:
# List all generated future returns files
returns_files = glob.glob(os.path.join(OUTPUT_PATH, "future_returns_*.pkl"))
returns_files.sort()

print(f"\n{'='*60}")
print(f"GENERATED FUTURE RETURNS FILES")
print(f"{'='*60}")
print(f"Total files: {len(returns_files)}\n")

if returns_files:
    total_size_bytes = 0
    
    for i, file in enumerate(returns_files, 1):
        file_size = os.path.getsize(file)
        file_size_mb = file_size / (1024*1024)
        total_size_bytes += file_size
        file_name = os.path.basename(file)
        
        print(f"{i:2d}. {file_name:<35} {file_size_mb:>8.2f} MB")
    
    total_size_mb = total_size_bytes / (1024*1024)
    total_size_gb = total_size_mb / 1024
    
    print(f"\nTotal size: {total_size_mb:,.1f} MB ({total_size_gb:.2f} GB)")
    
    if returns_files:
        print(f"\nFirst file: {os.path.basename(returns_files[0])}")
        print(f"Last file: {os.path.basename(returns_files[-1])}")
        print(f"Average file size: {total_size_mb/len(returns_files):.2f} MB")
else:
    print("No future returns files found.")
    print(f"Output directory: {OUTPUT_PATH}")
    print(f"Check if processing was completed successfully.")


GENERATED FUTURE RETURNS FILES
Total files: 45

 1. future_returns_2022-01.pkl             18.99 MB
 2. future_returns_2022-02.pkl             17.40 MB
 3. future_returns_2022-03.pkl             19.75 MB
 4. future_returns_2022-04.pkl             19.53 MB
 5. future_returns_2022-05.pkl             20.56 MB
 6. future_returns_2022-06.pkl             20.05 MB
 7. future_returns_2022-07.pkl             20.72 MB
 8. future_returns_2022-08.pkl             20.83 MB
 9. future_returns_2022-09.pkl             20.77 MB
10. future_returns_2022-10.pkl             21.98 MB
11. future_returns_2022-11.pkl             21.46 MB
12. future_returns_2022-12.pkl             22.18 MB
13. future_returns_2023-01.pkl             22.39 MB
14. future_returns_2023-02.pkl             21.41 MB
15. future_returns_2023-03.pkl             25.51 MB
16. future_returns_2023-04.pkl             26.05 MB
17. future_returns_2023-05.pkl             28.17 MB
18. future_returns_2023-06.pkl             27.76 MB
19. future_retu

## Usage Instructions

### How to use this notebook:

1. **Single Month Processing**: Use `process_monthly_returns('2025-02')` to process one month

2. **Batch Processing**: Use `batch_process_returns('2024-01', '2024-12')` to process a range

3. **Load Generated Data**: 
   ```python
   import pandas as pd
   df = pd.read_pickle('/home/craz/crypto/crypto-data/future_returns/future_returns_2025-02.pkl')
   ```

### Data Structure:
- **Index**: MultiIndex (open_time, symbol)
- **Columns**: return_1h, return_2h, return_3h, ..., return_144h
- **Values**: Future returns calculated as (future_price / current_price) - 1

### Example Return Calculation:
- Symbol at 13:00 with 2h return = (15:00 open_price / 13:00 open_price) - 1
- Positive return = price increased
- Negative return = price decreased
- NaN = future data not available